## Reading the Feature selection file data

In [88]:
# Function to read the input file
import pandas as pd

# Note: Please change the path here for the input file you want to read.
# Read column names from the .names file
df_names = pd.read_csv('./dataset/question_2/wbcd/wbcd.names', sep= ':', skiprows=1, header=None)
feature_list = df_names[0].tolist()

# Note: Please change the path here for the input file you want to read.
column_list = feature_list + ['class_label']
# Read data from the .data file
df_data = pd.read_csv('./dataset/question_2/wbcd/wbcd.data', names=column_list, index_col=False)

# Discretise features once for FilterGA
NUMBER_OF_BINS = 2

for feature in feature_list:
    df_data[feature] = pd.qcut(
        df_data[feature],
        q=NUMBER_OF_BINS,
        labels=False,
        duplicates='drop'
    )

In [89]:
# Now, we have read the data file successfully and assign header too after reading the name file
df_data.head(5)

,feature1,feature2,feature3,feature4,feature5,feature6,feature7,feature8,feature9,feature10,...,feature22,feature23,feature24,feature25,feature26,feature27,feature28,feature29,feature30,class_label
0,1,0,1,1,1,1,1,1,1,1,...,0,1,1,1,1,1,1,1,1,2
1,1,0,1,1,0,0,1,1,1,0,...,0,1,1,0,0,1,1,0,1,2
2,1,1,1,1,1,1,1,1,1,0,...,1,1,1,1,1,1,1,1,1,2
3,0,1,0,0,1,1,1,1,1,1,...,1,1,0,1,1,1,1,1,1,2
4,1,0,1,1,1,1,1,1,1,0,...,0,1,1,1,0,1,1,0,0,2


## Importing the Required packages

In [90]:
import random
import math
import numpy as np
import time
from deap import base, creator, tools

In [91]:
# Mantaining the size of individual chromosome to be equal to the number of features present in the dataset
IND_SIZE = len(feature_list)
# Defining the Population size for the Genetic Algorithm
POPULATION_SIZE = 30
# Defining the number of generations for the Genetic Algorithm
NUM_OF_GENERATION = 50
# Defining the mutation rate for the child to be mutated
MUTATION_RATE = 0.20
# Calculating the elite size based on the population size, ensuring it is at least 2
ELITE_SIZE = max(2, int(0.05*POPULATION_SIZE))

### Helper functions

In [92]:
# Function to perform mutation on a child individual
def mutate_child(child):
    if random.random() < MUTATION_RATE:
        # Perform flip mutation on the child
        index = random.randrange(len(child))
        child[index] = 1 - child[index]
    return child

# Purpose of this function is that when we supply an array to it, it will give an object with count 
def get_class_count(label_array):
    count_object = {};
    # Iterating over the label array and then counting the number of times each element is present in the array and then storing it in the count_object
    for element in label_array:
        if count_object.get(element):
            count_object[element] = count_object.get(element) + 1;
        else:
            count_object[element] = 1;
    return count_object;

# Function to Calculate Entropy = ["Y", "N", "Y", "N"]
def calculate_entropy(label_array):
    # After this step, we get: {A: 2, B: 2, C: 3}
    label_count_obj = get_class_count(label_array);
    
    # Let's get the number of total rows
    total_prediction = len(label_array);

    entropy = 0;

    # Iterating over the count of the elements and then finally calculating the entropy
    for element in label_count_obj:
        prob_element = (label_count_obj[element]/total_prediction);
        entropy -= (prob_element * math.log2(prob_element));

    return entropy;

# Function to calculate the information gain for a particular feature, which will take the parent label list and the child label list and then it will return the information gain for that feature
def calculate_information_gain(parent_y_list, child_y_list):
    # Calculating Parent Entropy
    parent_entropy = calculate_entropy(parent_y_list);

    # Calculating Child Weighted Entropy
    child_weighted_entropy = calculate_weighted_entropy(child_y_list, parent_y_list);

    # Information gain will be subtracted from parent entropy
    return parent_entropy - child_weighted_entropy;

# Function to calculate the weighted entropy for a particular feature, which will take the child label list and the parent label list and then it will return the weighted entropy for that feature
def calculate_weighted_entropy(child_y_list, parent_y_list):
    weighted_entropy = 0;

    # Iterating over each child label list
    for child_y_sublist in child_y_list:
        # Calculating entropy of each child label list
        child_entropy = calculate_entropy(child_y_sublist);
        # Calculating weighted entropy using child entropy
        weighted_entropy += (len(child_y_sublist)/len(parent_y_list))*child_entropy;
    
    # Returning calculated Weighted Entropy
    return weighted_entropy;

# Function to get selected feature list from individual chromosome list
def get_selected_features(individual):
     # List to store selected_feature list
    selected_features = list()

    # 1. Iterating over the individual chromosome to found the feature selected or not
    for index, item in enumerate(individual):
        # This mean this feature is selected
        if item == 1:
            selected_features.append(feature_list[index])

    return selected_features

# Filter based: Evaluation Strategy for each individual chromosome in the population
def do_evaluation(individual):
   
    # 1. Iterating over the individual chromosome to found the feature selected or not
    selected_features = get_selected_features(individual)

    if(len(selected_features) == 0):
        return (0,)
    
    # 2: Extracting data frame with the selected features
    filtered_df = df_data[selected_features + ['class_label']].copy()


    # 3: Create groups of class labels based on selected feature combinations
    grouped_y = (
        filtered_df
        .groupby(selected_features, observed=True)['class_label']
        .apply(list)
        .tolist()
    )
 
    # 4: Calculate Information Gain
    parent_y = filtered_df['class_label'].tolist()
    information_gain = calculate_information_gain(
        parent_y,
        grouped_y
    )
   
    return (information_gain,)
    
# Function to calculate avg of fitness among top n individual
def calculate_avg_fitness(population, n):
    # Selecting the top n individuals based on their fitness values
    top_n_individual_in_gen = tools.selBest(population, n)
    top_n_individual_fitness_avg = 0
    
    for individual in top_n_individual_in_gen:
        top_n_individual_fitness_avg += individual.fitness.values[0] 
    
    return (top_n_individual_fitness_avg / n)


### Registering Functions and Creating Types

In [93]:
# Creating Fitness and Individual Class
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

# Registering function with the toolbox
toolbox = base.Toolbox()

# Defining the Individual and Population structure for the Genetic Algorithm
toolbox.register("attr_binary", random.choice, [0, 1])
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_binary, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
# Evaluation: Custom Method for evaluation
toolbox.register("evaluate", do_evaluation)
# Selection: K-Tournament approach
toolbox.register("select", tools.selTournament, tournsize=3)
# Crossover: One Point crossover approach
toolbox.register("crossover", tools.cxOnePoint)
# Mutation: Custom method with 20% Mutation rate for each individual
toolbox.register('mutate', mutate_child)

/opt/anaconda3/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/opt/anaconda3/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


## Classifier for Checking Accuracy of the Best Individual

In [94]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Helper function to check accuracy for a dataframe with selected feature list
def classify_selected_features(df, selected_features):
    # Extract selected features
    X = df[selected_features]

    # Extract class label
    y = df['class_label']

    # Create classifier
    classifier = GaussianNB()

    # Train classifier on the entire dataset
    classifier.fit(X, y)

    # Predict on the same dataset
    y_pred = classifier.predict(X)

    # Calculate training accuracy
    accuracy = accuracy_score(y, y_pred)

    return accuracy

### Evolutionary Algorithm 

In [ ]:
# Seed list for 5 random GA runs
ga_seeds_list = [42, 52, 62, 72, 82]

# A list of list storing fitness of best 5 individual in each generation for each ga runs
convergence_data = list();

# A list storing computation time for each ga run
filter_ga_times = []

# Iterating through each GA run with different seeds
for ga_index, ga_seed in enumerate(ga_seeds_list):
    # Setting the seed using an integer
    random.seed(ga_seed)

    convergence_data.append([]);
    
    # Initializing a initial population
    initial_population = toolbox.population(POPULATION_SIZE)

    # Evaluating fitness for each individual
    for individual in initial_population:
            individual.fitness.values = toolbox.evaluate(individual)

    # Counter to store the computational time for Filter based GA
    # Storing the start time
    start_time = time.perf_counter()
    
    # Iterating through each generation for the current GA run
    for gen_index in range(NUM_OF_GENERATION):
        # Creating a new population for the next generation
        new_population = list()
        
        # Performing Elitism so that we have the best individual carried forward
        elites = tools.selBest(initial_population, ELITE_SIZE)
        elites = list(map(toolbox.clone, elites))
        new_population.extend(elites)

        # Till the new population is filled to the required size, perform selection, crossover, and mutation
        while(len(new_population)<POPULATION_SIZE):
            # Selection: K-Tournament approach - choose two individual for crossover
            offspring = toolbox.select(individuals = initial_population, k = 2)
            # Clone the selected individuals
            [parent1, parent2] = map(toolbox.clone, offspring)
            
            # Crossover: One Point crossover approach
            [child1, child2] = toolbox.crossover(parent1, parent2)
            
            # Mutation: Custom method with 20% Mutation rate for each individual
            child1 = toolbox.mutate(child1)
            child2 = toolbox.mutate(child2)

            # Evaluating the fitness of the newly created children
            child1.fitness.values = toolbox.evaluate(child1)
            child2.fitness.values = toolbox.evaluate(child2)

            # Adding only if the new population is not yet filled to the required size
            if (len(new_population) < POPULATION_SIZE):
                new_population.append(child1)
                
            if (len(new_population) < POPULATION_SIZE):
                new_population.append(child2)

        # Updating the initial population for the next generation
        initial_population = new_population

        # Calculating the average fitness of the top 5 individuals in the current generation and storing it for convergence analysis
        top_5_individual_fitness_avg = calculate_avg_fitness(new_population, 5)
        # Adding data to convergence_data for the current GA run and generation
        convergence_data[ga_index].append(top_5_individual_fitness_avg)
        # DEBUG LOG(Uncomment to see result): Log to print result for each generation of a GA Run
        print(f'GA Index: {ga_index + 1}, Generation Index: {gen_index + 1}, Top 5 individual avg fitness: {convergence_data[ga_index][gen_index]}')

    # Counter to store the computational time for Filter based GA
    # Storing the end time    
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    filter_ga_times.append(execution_time)

    # Finding best inidvidual for a particulat GA run and finding modal accuracy for that
    best_individual = tools.selBest(new_population, 1)[0]
    selected_features = get_selected_features(best_individual)
    accuracy = classify_selected_features(df_data, selected_features)

    # DEBUG LOG(Uncomment to see result): Printing the best individual and its fitness for the current GA run 
    print(f'GA Index: {ga_index + 1}, Fitness value: {best_individual.fitness.values}, Accuracy: {accuracy}')

GA Index: 1, Generation Index: 1, Top 5 individual avg fitness: 0.9409496698542898
GA Index: 1, Generation Index: 2, Top 5 individual avg fitness: 0.9434962823315611
GA Index: 1, Generation Index: 3, Top 5 individual avg fitness: 0.9477142085178528
GA Index: 1, Generation Index: 4, Top 5 individual avg fitness: 0.9491201839132835
GA Index: 1, Generation Index: 5, Top 5 individual avg fitness: 0.9491201839132835
GA Index: 1, Generation Index: 6, Top 5 individual avg fitness: 0.9491201839132835
GA Index: 1, Generation Index: 7, Top 5 individual avg fitness: 0.9491201839132835
GA Index: 1, Generation Index: 8, Top 5 individual avg fitness: 0.9491201839132835
GA Index: 1, Generation Index: 9, Top 5 individual avg fitness: 0.9491201839132835
GA Index: 1, Generation Index: 10, Top 5 individual avg fitness: 0.9491201839132835
GA Index: 1, Generation Index: 11, Top 5 individual avg fitness: 0.9491201839132835
GA Index: 1, Generation Index: 12, Top 5 individual avg fitness: 0.9491201839132835
G

### Convergence Curve Plotting and Data Calculation

In [ ]:
# Calculating the average fitness of best 5 individuals across generations for all GA runs
x_list = np.arange(1, NUM_OF_GENERATION + 1)
y_list = []

# Iterating through each generation
for col in range(NUM_OF_GENERATION):
    # Calculating the sum of fitness values
    sum_across_each_generation = 0
    # Iterating through each GA run to calculate the sum of fitness values for the current generation
    for ga_run in range(len(convergence_data)):
        sum_across_each_generation += convergence_data[ga_run][col]
    y_list.append(sum_across_each_generation / len(convergence_data))

In [ ]:
# Plotting the convergence curve for the average fitness of best 5 individuals across generations
import matplotlib.pyplot as plt

# Plot convergence curve
plt.figure(figsize=(10, 6))

plt.plot(x_list, y_list)

plt.xlabel("Generation")
plt.ylabel("Average Fitness of Best 5 Individuals")
plt.title("GA Convergence Curve")

plt.grid(True)
plt.show()

In [ ]:
filter_ga_times